# Point of this Notebook

A long-term archive of Gold features that is **readable by Apache Iceberg engines** -> Snowflake,
Trino, Athena, Spark OSS, etc. while living side-by-side with your Delta tables in the same Unity
Catalog.

We build this with **Delta UniForm** (`delta.universalFormat.enabledFormats = 'iceberg'`): the table
is stored as Delta, and Databricks automatically generates Iceberg metadata on top of it so external
Iceberg clients can read it through Unity Catalog's Iceberg REST catalog endpoint.

## Create the Iceberg-readable (UniForm) table

In [0]:
CATALOG = "raghavan_trading_signals"
ICEBERG = f"{CATALOG}.archive.market_features_iceberg"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ICEBERG} (
    symbol STRING, trade_date DATE,
    open DOUBLE, high DOUBLE, low DOUBLE, close DOUBLE, adj_close DOUBLE, volume BIGINT,
    daily_return DOUBLE, daily_return_pct DOUBLE, log_return DOUBLE,
    daily_range DOUBLE, overnight_gap DOUBLE,
    sma_7 DOUBLE, sma_14 DOUBLE, sma_50 DOUBLE, sma_200 DOUBLE,
    rsi_14 DOUBLE, macd_line DOUBLE, macd_signal DOUBLE, macd_histogram DOUBLE,
    bb_upper DOUBLE, bb_middle DOUBLE, bb_lower DOUBLE, bb_width DOUBLE, bb_position DOUBLE,
    volatility_10d DOUBLE, volatility_20d DOUBLE, volatility_60d DOUBLE,
    volume_ratio DOUBLE, vix DOUBLE, yield_curve_slope DOUBLE, fed_funds_rate DOUBLE,
    archive_timestamp TIMESTAMP, archive_batch_id STRING,
    trade_year INT GENERATED ALWAYS AS (YEAR(trade_date))
)
USING DELTA
PARTITIONED BY (trade_year)
TBLPROPERTIES (
    'delta.columnMapping.mode'             = 'name',
    'delta.enableIcebergCompatV2'          = 'true',
    'delta.universalFormat.enabledFormats' = 'iceberg'
)
COMMENT 'Archived Gold features (Delta + UniForm Iceberg) for cross-engine compliance access'
""")
print("Iceberg-readable (UniForm) table ready.")

> **The three TBLPROPERTIES are the whole trick.** `delta.universalFormat.enabledFormats = 'iceberg'`
> turns on automatic Iceberg metadata generation; it requires `delta.enableIcebergCompatV2 = 'true'`
> and `delta.columnMapping.mode = 'name'`. `trade_year` is a **generated column**, that Delta computes it
> automatically from `trade_date`, so you never write to it directly.

## Idempotent archive via MERGE

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from datetime import datetime, timedelta

def run_archive_job(cutoff_days=30):
    cutoff = (datetime.now() - timedelta(days=cutoff_days)).strftime("%Y-%m-%d")
    batch_id = datetime.now().strftime("archive_%Y%m%d_%H%M%S")
    cols = ["symbol","trade_date","open","high","low","close","adj_close","volume",
            "daily_return","daily_return_pct","log_return","daily_range","overnight_gap",
            "sma_7","sma_14","sma_50","sma_200","rsi_14","macd_line","macd_signal",
            "macd_histogram","bb_upper","bb_middle","bb_lower","bb_width","bb_position",
            "volatility_10d","volatility_20d","volatility_60d","volume_ratio","vix",
            "yield_curve_slope","fed_funds_rate"]
    src = (spark.table(f"{CATALOG}.gold.daily_features")
           .filter(col("trade_date") < cutoff).select(*cols)
           .withColumn("archive_timestamp", current_timestamp())
           .withColumn("archive_batch_id", lit(batch_id)))
    src.createOrReplaceTempView("new_archive_data")

    # Explicitly list insert columns and OMIT the generated `trade_year` — Delta computes it.
    # `INSERT *` would expand to every target column (incl. trade_year)
    insert_cols = cols + ["archive_timestamp", "archive_batch_id"]
    col_list = ", ".join(insert_cols)
    val_list = ", ".join(f"s.{c}" for c in insert_cols)
    spark.sql(f"""
        MERGE INTO {ICEBERG} AS t
        USING new_archive_data AS s
        ON t.symbol = s.symbol AND t.trade_date = s.trade_date
        WHEN NOT MATCHED THEN INSERT ({col_list}) VALUES ({val_list})
    """)
    return spark.table(ICEBERG).count()

print("Total archived rows:", f"{run_archive_job(30):,}")

## Inspecting metadata

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {ICEBERG}"))   # versions/commits (Delta analog of .snapshots)

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {ICEBERG}"))     # format, location, partition columns

In [0]:
# Partition layout (Delta analog of .partitions)
display(spark.sql(f"""
    SELECT trade_year, COUNT(*) AS n
    FROM {ICEBERG}
    GROUP BY trade_year ORDER BY trade_year
"""))


In [0]:
# Same Unity Catalog, Delta active + Iceberg-readable archive, queried identically
display(spark.sql(f"""
    SELECT 'Delta (active)' AS source, COUNT(*) n, MIN(trade_date) lo, MAX(trade_date) hi
    FROM {CATALOG}.gold.daily_features
    UNION ALL
    SELECT 'Iceberg (archive)', COUNT(*), MIN(trade_date), MAX(trade_date)
    FROM {ICEBERG}
"""))

## Validation: Iceberg table

In [0]:
# 1. Confirm UniForm/Iceberg is ENABLED on the table (config check)
props = spark.sql(f"SHOW TBLPROPERTIES {ICEBERG}").toPandas()
display(props[props["key"].str.contains(
    "universalFormat|iceberg|columnMapping|enableIcebergCompat", case=False, na=False)])